# ✂️ Train/Test Split & Data Leakage
**One-line description:** Properly partition your data to get honest model evaluation and avoid the silent killer — data leakage.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/04_train_test_split.ipynb)


In [ ]:
# Install required libraries (run this cell first in Google Colab)
!pip install scikit-learn pandas numpy matplotlib seaborn --quiet


## 📖 What is Train/Test Split?

Train/test split divides your dataset into at least two parts:
- **Training set**: used to fit/train the model
- **Test set**: held out, used only for final evaluation

**Analogy:** Think of it like preparing for an exam. Your textbook exercises are your training data — you practice on them. The actual exam questions are your test data — they're new, unseen problems. If you peek at the exam questions while studying (data leakage), your exam score won't reflect your true knowledge.

### Common Splits:
| Split | Training | Validation | Test |
|-------|---------|-----------|------|
| 80/20 | 80% | — | 20% |
| 70/30 | 70% | — | 30% |
| 60/20/20 | 60% | 20% | 20% |


## 💡 Why Does It Matter?

- **Overfitting detection**: a model that memorizes training data will fail on test data
- **Honest evaluation**: test set simulates real-world deployment performance
- **Data leakage** causes artificially inflated metrics — your model looks great in testing but fails in production
- **Stratified splits** preserve class ratios — critical for imbalanced datasets


## ⚙️ How Does It Work?

Strategies we'll cover:
1. **Random Split** — simple, for i.i.d. data
2. **Stratified Split** — preserves class proportions
3. **Time-Series Split** — respects temporal ordering (no future leakage)
4. **Data Leakage** — what it is, how it sneaks in, how to prevent it
5. **sklearn Pipeline** — the correct way to chain preprocessing + modeling
6. **Cross-Validation** — more robust than a single split


## 🛠️ Hands-on Code

### Step 1: Create an Imbalanced Classification Dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     TimeSeriesSplit, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Create an imbalanced dataset (80% class 0, 20% class 1)
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    weights=[0.80, 0.20],  # class imbalance
    random_state=42
)

df = pd.DataFrame(X, columns=[f'Feature_{i+1}' for i in range(20)])
df['Target'] = y

print("Dataset shape:", df.shape)
print(f"\nClass distribution:")
print(df['Target'].value_counts())
print(f"\nClass balance: {df['Target'].value_counts(normalize=True).round(3).to_dict()}")


In [ ]:
# --- Visualization 1: Class imbalance visualization ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Class distribution bar chart
class_counts = df['Target'].value_counts()
axes[0].bar(['Class 0 (Majority)', 'Class 1 (Minority)'],
            class_counts.values,
            color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Class 0 (80%)', 'Class 1 (20%)'],
            autopct='%1.1f%%', colors=['steelblue', 'coral'],
            startangle=90, explode=(0, 0.1))
axes[1].set_title('Class Balance Ratio', fontsize=12, fontweight='bold')

# Scatter of first 2 features colored by class
scatter = axes[2].scatter(X[:500, 0], X[:500, 1], c=y[:500],
                           cmap='coolwarm', alpha=0.5, s=15)
axes[2].set_title('Feature 1 vs Feature 2\n(colored by class)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Feature 1')
axes[2].set_ylabel('Feature 2')
plt.colorbar(scatter, ax=axes[2], label='Class')

plt.suptitle('Imbalanced Classification Dataset Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 2: Random vs. Stratified Split


In [ ]:
features = [c for c in df.columns if c != 'Target']
X_data = df[features].values
y_data = df['Target'].values

# --- Random split ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)

# --- Stratified split ---
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

print("=" * 55)
print(f"{'Split Type':<20} {'Train Class1%':>15} {'Test Class1%':>15}")
print("=" * 55)

train_r_pct = y_train_r.mean() * 100
test_r_pct = y_test_r.mean() * 100
print(f"{'Random Split':<20} {train_r_pct:>14.1f}% {test_r_pct:>14.1f}%")

train_s_pct = y_train_s.mean() * 100
test_s_pct = y_test_s.mean() * 100
print(f"{'Stratified Split':<20} {train_s_pct:>14.1f}% {test_s_pct:>14.1f}%")
print("=" * 55)
print(f"\nOriginal class 1 proportion: {y_data.mean()*100:.1f}%")
print("\nStratified split preserves the original class ratio in both sets!")


### Step 3: Time-Series Split (No Future Leakage)


In [ ]:
# For time-series data, you CANNOT randomly shuffle — that would use future info to predict the past!
# TimeSeriesSplit always uses past data for training and future data for testing

tscv = TimeSeriesSplit(n_splits=5)

fig, axes = plt.subplots(1, 1, figsize=(12, 5))
colors_train = ['steelblue', 'royalblue', 'navy', 'dodgerblue', 'cornflowerblue']
colors_test = ['coral', 'tomato', 'red', 'orangered', 'darkorange']

n_samples_ts = 1000
for i, (train_idx, test_idx) in enumerate(tscv.split(range(n_samples_ts))):
    axes.scatter(train_idx, [i] * len(train_idx), c=colors_train[i], s=2, label=f'Fold {i+1} Train' if i==0 else "")
    axes.scatter(test_idx,  [i] * len(test_idx),  c=colors_test[i],  s=2, label=f'Fold {i+1} Test' if i==0 else "")
    axes.text(n_samples_ts + 5, i, f'Fold {i+1}', va='center', fontsize=9)

axes.set_xlabel('Sample Index (time →)')
axes.set_ylabel('Fold')
axes.set_title('TimeSeriesSplit: Train always before Test (no future leakage)', fontsize=13, fontweight='bold')
axes.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('timeseries_split.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 4: Data Leakage — The Silent Model Killer


In [ ]:
# DATA LEAKAGE DEMO: What happens when you fit scaler on ALL data (wrong way)?

scaler_leaky = StandardScaler()
X_all_scaled = scaler_leaky.fit_transform(X_data)  # FIT ON ALL DATA = LEAKAGE!
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_all_scaled, y_data, test_size=0.2, random_state=42, stratify=y_data
)

# CORRECT WAY: fit scaler only on training data
scaler_correct = StandardScaler()
X_train_correct = scaler_correct.fit_transform(X_train_s)  # fit+transform on train
X_test_correct  = scaler_correct.transform(X_test_s)        # only transform on test

# Train same model on both versions
lr_leaky = LogisticRegression(random_state=42, max_iter=1000)
lr_leaky.fit(X_train_leak, y_train_leak)
acc_leaky = accuracy_score(y_test_leak, lr_leaky.predict(X_test_leak))

lr_correct = LogisticRegression(random_state=42, max_iter=1000)
lr_correct.fit(X_train_correct, y_train_correct := y_train_s)
acc_correct = accuracy_score(y_test_s, lr_correct.predict(X_test_correct))

print("⚠️  DATA LEAKAGE COMPARISON:")
print(f"  Leaky (scaler fit on all data):  {acc_leaky:.4f} accuracy  ← INFLATED, dishonest!")
print(f"  Correct (scaler fit on train):   {acc_correct:.4f} accuracy  ← HONEST estimate")
print()
print("Even small leakage can cause you to select the wrong model in production!")


In [ ]:
# THE RIGHT WAY: Use sklearn Pipeline to prevent leakage automatically
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# Pipeline correctly fits scaler only on training data during cross-validation
cv_scores = cross_val_score(pipeline, X_data, y_data, cv=StratifiedKFold(n_splits=5),
                             scoring='accuracy', n_jobs=-1)

print("Cross-validation with Pipeline (leak-proof):")
print(f"  Fold scores: {cv_scores.round(4)}")
print(f"  Mean CV accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print()
print("Pipeline automatically applies: fit_transform on train, transform on test for each fold")


In [ ]:
# --- Visualization 2: Cross-validation fold scores ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV fold scores
axes[0].bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {cv_scores.mean():.4f}')
axes[0].fill_between(range(0, 7),
                      cv_scores.mean() - cv_scores.std(),
                      cv_scores.mean() + cv_scores.std(),
                      alpha=0.2, color='red', label=f'±1 Std: {cv_scores.std():.4f}')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('5-Fold Stratified CV Accuracy
(using Pipeline — leak-proof)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)

# Correct vs leaky comparison
axes[1].bar(['Leaky
(fit on all)', 'Correct
(fit on train)', 'CV Mean'],
            [acc_leaky, acc_correct, cv_scores.mean()],
            color=['red', 'steelblue', 'green'], edgecolor='black')
axes[1].set_title('Accuracy: Leaky vs Correct vs CV', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.5, 1.0)
for i, v in enumerate([acc_leaky, acc_correct, cv_scores.mean()]):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

plt.suptitle('Train/Test Split and Data Leakage Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cv_analysis.png', dpi=100, bbox_inches='tight')
plt.show()


## 📌 Key Takeaways

- Always use **stratified split** for classification tasks — preserves class ratios
- For **time-series data**, never shuffle — use `TimeSeriesSplit`
- **Data leakage** occurs when information from the test set influences training — always inflates metrics
- Common leakage sources: scaling on all data, imputing on all data, feature selection on all data
- Use **sklearn Pipeline** to automatically prevent leakage in cross-validation
- **Cross-validation** gives a more reliable estimate than a single train/test split
- The golden rule: **fit on train, transform on test** — never the reverse
